# Robustness Check — Class Weighting vs. SMOTE vs. SMOTE-ENN
Same 80:20 stratified split and 26 features as `analysis.ipynb` / `RandomForest.ipynb`.
Resampling is applied to the training fold only; the held-out test set is never touched.

In [1]:
import pandas as pd
import numpy as np
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, average_precision_score
)

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

df = pd.read_csv("online_shoppers_intention.csv")
NOMINAL_INT_COLS = ["OperatingSystems", "Browser", "Region", "TrafficType"]
df[NOMINAL_INT_COLS] = df[NOMINAL_INT_COLS].astype("category")
df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)

X = df_encoded.drop("Revenue", axis=1)
y = df_encoded["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def evaluate(y_true, y_pred, y_prob):
    return {
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "AUC": round(roc_auc_score(y_true, y_prob), 4),
        "PR-AUC": round(average_precision_score(y_true, y_prob), 4),
        "Precision(Purchase)": round(precision_score(y_true, y_pred), 4),
        "Recall(Purchase)": round(recall_score(y_true, y_pred), 4),
        "F1(Purchase)": round(f1_score(y_true, y_pred), 4),
        "MCC": round(matthews_corrcoef(y_true, y_pred), 4),
    }

results = []

In [3]:
lr_base = LogisticRegression(class_weight='balanced', max_iter=20000, random_state=42)
lr_base.fit(X_train_scaled, y_train)
p, pr = lr_base.predict(X_test_scaled), lr_base.predict_proba(X_test_scaled)[:, 1]
results.append({"Model": "Logistic Regression", "Variant": "Baseline (class_weight)", **evaluate(y_test, p, pr)})

rf_base = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1)
rf_base.fit(X_train_scaled, y_train)
p, pr = rf_base.predict(X_test_scaled), rf_base.predict_proba(X_test_scaled)[:, 1]
results.append({"Model": "Random Forest", "Variant": "Baseline (class_weight)", **evaluate(y_test, p, pr)})

In [4]:
smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_train_scaled, y_train)

lr_sm = LogisticRegression(max_iter=20000, random_state=42)
lr_sm.fit(X_sm, y_sm)
p, pr = lr_sm.predict(X_test_scaled), lr_sm.predict_proba(X_test_scaled)[:, 1]
results.append({"Model": "Logistic Regression", "Variant": "SMOTE", **evaluate(y_test, p, pr)})

rf_sm = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_sm.fit(X_sm, y_sm)
p, pr = rf_sm.predict(X_test_scaled), rf_sm.predict_proba(X_test_scaled)[:, 1]
results.append({"Model": "Random Forest", "Variant": "SMOTE", **evaluate(y_test, p, pr)})

In [5]:
smote_enn = SMOTEENN(random_state=42)
X_se, y_se = smote_enn.fit_resample(X_train_scaled, y_train)

lr_se = LogisticRegression(max_iter=20000, random_state=42)
lr_se.fit(X_se, y_se)
p, pr = lr_se.predict(X_test_scaled), lr_se.predict_proba(X_test_scaled)[:, 1]
results.append({"Model": "Logistic Regression", "Variant": "SMOTE-ENN", **evaluate(y_test, p, pr)})

rf_se = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_se.fit(X_se, y_se)
p, pr = rf_se.predict(X_test_scaled), rf_se.predict_proba(X_test_scaled)[:, 1]
results.append({"Model": "Random Forest", "Variant": "SMOTE-ENN", **evaluate(y_test, p, pr)})

In [6]:
res_df = pd.DataFrame(results)
res_df.to_csv("smote_diagnostic_results.csv", index=False)
res_df

,Model,Variant,Accuracy,AUC,PR-AUC,Precision(Purchase),Recall(Purchase),F1(Purchase),MCC
0,Logistic Regression,Baseline (class_weight),0.8366,0.8918,0.6208,0.4821,0.7382,0.5832,0.5042
1,Random Forest,Baseline (class_weight),0.8861,0.9241,0.7221,0.6135,0.7147,0.6602,0.5947
2,Logistic Regression,SMOTE,0.8455,0.8903,0.6234,0.5009,0.7304,0.5942,0.5165
3,Random Forest,SMOTE,0.8869,0.9189,0.6888,0.6157,0.7173,0.6626,0.5976
4,Logistic Regression,SMOTE-ENN,0.7993,0.8953,0.6146,0.4250,0.8377,0.5639,0.4949
5,Random Forest,SMOTE-ENN,0.8625,0.9139,0.6402,0.5373,0.8115,0.6465,0.5841


## SHAP ranking stability: baseline RF vs. SMOTE RF

In [7]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

X_sample = X_test_scaled_df.sample(n=min(500, len(X_test_scaled_df)), random_state=42)

explainer_base = shap.TreeExplainer(rf_base, feature_perturbation="tree_path_dependent")
sv_base = explainer_base.shap_values(X_sample, check_additivity=False)[:, :, 1]
imp_base = pd.Series(np.abs(sv_base).mean(axis=0), index=X.columns).sort_values(ascending=False)

explainer_sm = shap.TreeExplainer(rf_sm, feature_perturbation="tree_path_dependent")
sv_sm = explainer_sm.shap_values(X_sample, check_additivity=False)[:, :, 1]
imp_sm = pd.Series(np.abs(sv_sm).mean(axis=0), index=X.columns).sort_values(ascending=False)

print("Baseline RF — top 8 SHAP:\n", imp_base.head(8))
print("\nSMOTE RF — top 8 SHAP:\n", imp_sm.head(8))

Baseline RF — top 8 SHAP:
 PageValues                 0.230532
ExitRates                  0.039089
Month_Nov                  0.038438
BounceRates                0.026339
ProductRelated_Duration    0.023069
ProductRelated             0.017040
Month_May                  0.016057
Administrative             0.015598
dtype: float64

SMOTE RF — top 8 SHAP:
 PageValues                 0.242219
Month_Nov                  0.044578
Administrative_Duration    0.037029
ExitRates                  0.031131
ProductRelated_Duration    0.021821
Administrative             0.021807
TrafficType_2              0.019989
ProductRelated             0.017849
dtype: float64
